# Respiratory virus surveillance database

> **Study draft: all data are synthetic.** Run the cells yourself, inspect the generator, and personalize the analysis and report before using this for coursework. Simulated positivity rates are not clinical evidence.

This notebook rebuilds the SQLite database from a fixed seed, checks integrity and intentional data-quality conditions, and demonstrates two retrieval queries. It uses the Python standard library.

**Before you begin:** open Jupyter with this project folder as the working directory. The generation cell replaces `respiratory_surveillance.sqlite`.

## 1. Rebuild reproducibly

The seed controls randomized but structured dates, site mix, age bands, missing fields, tests, and repeat feed records. Inspect `generate_database.py` and change one modeling assumption after you understand its effect.

In [1]:
from pathlib import Path
import sqlite3
from generate_database import build_database, DEFAULT_SEED

PROJECT_DIR = Path.cwd()
DB_PATH = PROJECT_DIR / 'respiratory_surveillance.sqlite'
summary = build_database(DB_PATH, seed=DEFAULT_SEED)
print('Seed:', summary['seed'])
print('SQLite quick_check:', summary['quick_check'])
print('Foreign-key issues:', summary['foreign_key_issues'])
summary['counts']

Seed: 20261106
SQLite quick_check: ok
Foreign-key issues: 0


{'assay': 8,
 'lab_result': 10177,
 'participant': 1500,
 'participant_vaccination': 3451,
 'pathogen': 4,
 'raw_lab_feed': 10427,
 'region': 4,
 'site': 8,
 'specimen': 2400,
 'weekly_surveillance': 860}

## 2. Check keys, constraints, and row counts

SQLite foreign-key enforcement is connection-specific. Enable it on each notebook connection before checking or updating data. The generator also verifies foreign keys and the SQLite integrity check.

In [2]:
con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row
con.execute('PRAGMA foreign_keys = ON')
table_counts = {}
for row in con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"):
    table_counts[row['name']] = con.execute(f'SELECT COUNT(*) FROM "{row["name"]}"').fetchone()[0]
assert table_counts['specimen'] >= 1000
assert table_counts['lab_result'] >= 1000
assert con.execute('PRAGMA foreign_key_check').fetchall() == []
assert con.execute('PRAGMA quick_check').fetchone()[0] == 'ok'
table_counts

{'assay': 8,
 'lab_result': 10177,
 'participant': 1500,
 'participant_vaccination': 3451,
 'pathogen': 4,
 'raw_lab_feed': 10427,
 'region': 4,
 'site': 8,
 'specimen': 2400,
 'weekly_surveillance': 860}

## 3. Profile deliberate missing and duplicate data

The raw staging table keeps repeated source keys for a deduplication exercise. In the curated specimen table, an asymptomatic record has no onset date by definition; do not count that as a recording failure.

In [3]:
quality_queries = {
    'symptomatic onset missing': "SELECT COUNT(*) FROM specimen WHERE symptomatic=1 AND symptom_onset IS NULL",
    'symptomatic temperature missing': "SELECT COUNT(*) FROM specimen WHERE symptomatic=1 AND temperature_celsius IS NULL",
    'detected PCR missing Ct': "SELECT COUNT(*) FROM lab_result WHERE assay_code='PCR' AND result_category='detected' AND ct_value IS NULL",
    'detected PCR missing load': "SELECT COUNT(*) FROM lab_result WHERE assay_code='PCR' AND result_category='detected' AND viral_load_copies_ml IS NULL",
    'raw feed missing receipt time': "SELECT COUNT(*) FROM raw_lab_feed WHERE source_received_at IS NULL",
    'raw duplicate key groups': "SELECT COUNT(*) FROM (SELECT source_record_key FROM raw_lab_feed GROUP BY source_record_key HAVING COUNT(*)>1)",
}
for label, sql in quality_queries.items():
    print(f'{label:36s} {con.execute(sql).fetchone()[0]:>5,}')

duplicate_examples = con.execute("""
    SELECT source_record_key, COUNT(*) AS copies
    FROM raw_lab_feed GROUP BY source_record_key HAVING COUNT(*) > 1
    ORDER BY copies DESC, source_record_key LIMIT 10
""").fetchall()
[dict(row) for row in duplicate_examples]

symptomatic onset missing              322
symptomatic temperature missing        123
detected PCR missing Ct                 41
detected PCR missing load               56
raw feed missing receipt time          351
raw duplicate key groups               250


[{'source_record_key': 'EXT-00002-FLU_A-PCR-1', 'copies': 2},
 {'source_record_key': 'EXT-00030-FLU_A-PCR-1', 'copies': 2},
 {'source_record_key': 'EXT-00046-FLU_B-PCR-1', 'copies': 2},
 {'source_record_key': 'EXT-00059-FLU_B-PCR-1', 'copies': 2},
 {'source_record_key': 'EXT-00068-RSV-PCR-1', 'copies': 2},
 {'source_record_key': 'EXT-00074-SARS_COV2-PCR-1', 'copies': 2},
 {'source_record_key': 'EXT-00087-FLU_A-PCR-1', 'copies': 2},
 {'source_record_key': 'EXT-00091-FLU_A-PCR-1', 'copies': 2},
 {'source_record_key': 'EXT-00103-FLU_B-PCR-1', 'copies': 2},
 {'source_record_key': 'EXT-00108-SARS_COV2-PCR-1', 'copies': 2}]

## 4. Retrieve weekly surveillance signals

The view joins weekly counts to invented site, region, and pathogen labels. Interpret this only as a query demonstration: its thresholds and prevalence assumptions are educational simulation choices.

In [4]:
weekly_sql = """
SELECT week_start, region_name, site_name, pathogen_name,
       tests_count, positives_count, ROUND(100.0 * positivity_rate, 1) AS positivity_percent, alert_level
FROM v_outbreak_monitoring
WHERE alert_level IN ('watch', 'elevated') AND tests_count >= 10
ORDER BY positivity_rate DESC, week_start
LIMIT 15
"""
[dict(row) for row in con.execute(weekly_sql)]

[{'week_start': '2025-11-03',
  'region_name': 'South District',
  'site_name': 'South Urgent Care',
  'pathogen_name': 'Respiratory syncytial virus',
  'tests_count': 10,
  'positives_count': 6,
  'positivity_percent': 60.0,
  'alert_level': 'elevated'},
 {'week_start': '2026-01-05',
  'region_name': 'North District',
  'site_name': 'North Community Clinic',
  'pathogen_name': 'Influenza A',
  'tests_count': 10,
  'positives_count': 6,
  'positivity_percent': 60.0,
  'alert_level': 'elevated'},
 {'week_start': '2025-11-10',
  'region_name': 'North District',
  'site_name': 'North General Hospital',
  'pathogen_name': 'SARS-CoV-2',
  'tests_count': 20,
  'positives_count': 10,
  'positivity_percent': 50.0,
  'alert_level': 'elevated'},
 {'week_start': '2026-02-02',
  'region_name': 'East District',
  'site_name': 'East General Hospital',
  'pathogen_name': 'SARS-CoV-2',
  'tests_count': 12,
  'positives_count': 6,
  'positivity_percent': 50.0,
  'alert_level': 'elevated'},
 {'week_star

## 5. Compare positivity by age band and vaccination history

This example uses the first PCR replicate and a vaccination record dated no later than collection. It is a descriptive simulation query, not a vaccine-effectiveness analysis: the generator imposes artificial distributions and does not model real confounding or clinical outcomes.

In [5]:
age_vax_sql = """
SELECT p.age_band,
       CASE WHEN EXISTS (
           SELECT 1 FROM participant_vaccination AS v
           WHERE v.participant_id=p.participant_id
             AND v.pathogen_code='SARS_COV2'
             AND v.administered_on <= s.collected_on
       ) THEN 'recorded prior dose' ELSE 'no prior dose record' END AS vaccination_group,
       COUNT(*) AS tests,
       SUM(CASE WHEN lr.result_category='detected' THEN 1 ELSE 0 END) AS positives,
       ROUND(100.0 * AVG(CASE WHEN lr.result_category='detected' THEN 1.0 ELSE 0.0 END), 1) AS positivity_percent
FROM specimen AS s
JOIN participant AS p ON p.participant_id=s.participant_id
JOIN lab_result AS lr ON lr.specimen_id=s.specimen_id
WHERE lr.assay_code='PCR' AND lr.pathogen_code='SARS_COV2' AND lr.replicate_no=1
GROUP BY p.age_band, vaccination_group
ORDER BY CASE p.age_band
           WHEN 'under 5' THEN 1 WHEN '5-17' THEN 2 WHEN '18-49' THEN 3
           WHEN '50-64' THEN 4 WHEN '65+' THEN 5 END,
         vaccination_group
"""
[dict(row) for row in con.execute(age_vax_sql)]

[{'age_band': 'under 5',
  'vaccination_group': 'no prior dose record',
  'tests': 124,
  'positives': 19,
  'positivity_percent': 15.3},
 {'age_band': 'under 5',
  'vaccination_group': 'recorded prior dose',
  'tests': 73,
  'positives': 9,
  'positivity_percent': 12.3},
 {'age_band': '5-17',
  'vaccination_group': 'no prior dose record',
  'tests': 231,
  'positives': 27,
  'positivity_percent': 11.7},
 {'age_band': '5-17',
  'vaccination_group': 'recorded prior dose',
  'tests': 135,
  'positives': 18,
  'positivity_percent': 13.3},
 {'age_band': '18-49',
  'vaccination_group': 'no prior dose record',
  'tests': 627,
  'positives': 80,
  'positivity_percent': 12.8},
 {'age_band': '18-49',
  'vaccination_group': 'recorded prior dose',
  'tests': 367,
  'positives': 52,
  'positivity_percent': 14.2},
 {'age_band': '50-64',
  'vaccination_group': 'no prior dose record',
  'tests': 291,
  'positives': 36,
  'positivity_percent': 12.4},
 {'age_band': '50-64',
  'vaccination_group': 'reco